# weight-decay-l2-add — ex1: fold weight decay lambda*theta into the gradient

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `weight-decay-l2-add`. Running the final beacon cell reports progress against the `Optimizer: Weight decay L2` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Weight decay L2` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`weight-decay-l2-add`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "weight-decay-l2-add"
DD_SUBTOPIC = "Optimizer: Weight decay L2"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Weight decay (L2 fold) — quick refresher

L2 regularization adds `(lambda / 2) * ||theta||^2` to the loss. Its gradient w.r.t. `theta` is `lambda * theta`. Rather than build that into the loss expression, optimizers FOLD it directly into the gradient at each step:

```
if self.lmda != 0:
    g = g + self.lmda * theta
```

Then the rest of the optimizer treats this augmented `g` as the gradient. The `if self.lmda != 0` guard skips the fold (and the tensor allocation) when weight decay is disabled — a tiny but real performance win in the inner loop.

**This is the classical L2 form, NOT AdamW.** AdamW DECOUPLES the decay from the gradient — it subtracts `lr * lmda * theta` from `theta` directly, bypassing the moment estimates. Mixing them up is the single most common 'why does my Adam train weirdly' bug.

### Exercise 1 — fold weight decay lambda*theta into the gradient

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the L2 weight-decay fold `g = g + lmda * theta` inside an optimizer step, guarded by `if lmda != 0` to skip the fold when decay is disabled.
> Keywords: weight-decay, l2-fold, gradient-augment
> ```

**KCs targeted:** `weight-decay-l2-fold-into-grad`, `weight-decay-zero-lambda-bypass`

Implement `ex1_apply_weight_decay(theta, g, lmda)`. This is the TWO-LINE block from ARENA's SGD/RMSprop/Adam impls.

1. If `lmda == 0`, return `g` unchanged (the bypass — saves allocation in the inner loop).
2. Otherwise, return `g + lmda * theta` (do NOT mutate `g` in place — return a new tensor).

Inputs:
- `theta`: parameter tensor.
- `g`: gradient tensor, same shape as `theta`.
- `lmda`: float decay coefficient.

Output: the augmented gradient — same shape and dtype as `g`.

The test verifies BOTH paths: the augmented value when `lmda > 0`, the literal identity (same object) when `lmda == 0`, and the sign behavior when `theta` is negative.

In [ ]:
def ex1_apply_weight_decay(theta: Tensor, g: Tensor, lmda: float) -> Tensor:
    """Return g + lmda*theta when lmda != 0, else return g unchanged."""
    raise NotImplementedError()


def _test_ex1():
    # Basic positive theta, positive grad, positive lmda.
    theta = t.tensor([1.0, 2.0, 3.0])
    g     = t.tensor([0.1, 0.2, 0.3])
    out = ex1_apply_weight_decay(theta, g, lmda=0.01)
    expected = t.tensor([0.11, 0.22, 0.33])  # g + 0.01 * theta
    assert t.allclose(out, expected), (
        f'lmda=0.01: got {out}, expected {expected}; '
        f'check you are computing g + lmda * theta'
    )

    # === Bypass path: lmda == 0 must return g UNCHANGED ===
    g_for_bypass = t.tensor([0.5, -0.5, 0.0])
    out_bypass = ex1_apply_weight_decay(theta, g_for_bypass, lmda=0.0)
    assert t.equal(out_bypass, g_for_bypass), (
        f'lmda=0 must return g identical; got {out_bypass}'
    )
    # Ideally the SAME object — many ARENA solutions check this.
    assert out_bypass is g_for_bypass, (
        'lmda=0 should return the input g unchanged (same object), '
        'avoiding an unnecessary tensor allocation in the inner loop'
    )

    # === Negative theta — weight decay pulls toward zero ===
    theta_neg = t.tensor([-2.0, -1.0, 1.0, 2.0])
    g_zero = t.zeros(4)
    out_neg = ex1_apply_weight_decay(theta_neg, g_zero, lmda=0.1)
    expected_neg = t.tensor([-0.2, -0.1, 0.1, 0.2])
    assert t.allclose(out_neg, expected_neg), (
        f'with zero base gradient, fold should equal lmda * theta; '
        f'got {out_neg}, expected {expected_neg}'
    )
    # Sign: positive theta → positive augment → after update theta moves negative (toward 0).
    # Negative theta → negative augment → after update theta moves positive (toward 0).
    # This is the WHOLE POINT of weight decay.

    # === Input grad not mutated when lmda != 0 ===
    g_check = t.tensor([1.0, 2.0])
    g_snapshot = g_check.clone()
    _ = ex1_apply_weight_decay(t.tensor([10.0, 20.0]), g_check, lmda=0.5)
    assert t.equal(g_check, g_snapshot), (
        f'g was mutated by the function (now {g_check}, was {g_snapshot}); '
        f'this fold should be out-of-place; use g + lmda*theta, not g += lmda*theta'
    )

    # === Shape preservation on a multi-dim param ===
    theta_2d = t.randn(4, 5)
    g_2d = t.randn(4, 5)
    out_2d = ex1_apply_weight_decay(theta_2d, g_2d, lmda=0.01)
    assert out_2d.shape == (4, 5)
    assert t.allclose(out_2d, g_2d + 0.01 * theta_2d)
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_apply_weight_decay(theta, g, lmda):
    if lmda != 0:
        g = g + lmda * theta
    return g
```

**Why fold instead of adding to the loss.** Adding `(lmda/2)*||theta||^2` to the loss expression would work — but it would force autograd to track the regularizer through the backward, which is wasteful (the gradient of `(lmda/2)*||theta||^2` is just `lmda*theta`, computable directly). Folding skips the loss-side detour.

**Why the `if lmda != 0` guard.** In the inner training loop this runs once per param per step. With `lmda=0` (no decay), the addition `g + 0 * theta` would still allocate a new tensor the size of `theta` — pure waste. The branch costs one Python-level comparison and skips the allocation. For a model with 1000 parameter groups stepping 10k times, the saved allocations matter.

**Classical L2 fold ≠ AdamW.** AdamW subtracts `lr * lmda * theta` from `theta` directly AFTER the Adam update, bypassing the moment estimates entirely. The fold form (this drill) modifies `g`, so for Adam the decay gets scaled by `1 / (sqrt(v_hat) + eps)` along with everything else — usually undesirable for transformers. Knowing which form your library uses is critical.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()